# Install Libraries

In [2]:
!pip install -q pypdf
!pip install -q sentence-transformers
!pip install -q faiss-cpu
!pip install -q langchain-text-splitters
!pip install -q pymupdf pytesseract pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 78.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 65.6 MB/s eta 0:00:00:00:0100:01


In [3]:
import fitz
import pytesseract
from PIL import Image

print("PyMuPDF:", fitz.__doc__.split()[1])
print("Tesseract:", pytesseract.get_tesseract_version())

PyMuPDF: 1.28.2:
Tesseract: 4.1.1


In [4]:
pdf_paths = {

    "WHO": 
        "/kaggle/input/datasets/midhulaict/skinova-knowledge-base/Skinova_knowledge_base/WHO/who_common_skin_diseases.pdf",

    "DermNet": 
        "/kaggle/input/datasets/midhulaict/skinova-knowledge-base/Skinova_knowledge_base/Dermnet"
}

In [5]:
import os

pdf_files = []

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file.lower().endswith(".pdf"):
            pdf_files.append(os.path.join(root, file))

print("Total PDF files:", len(pdf_files))

for path in pdf_files:
    print(path)

Total PDF files: 8
/kaggle/input/datasets/midhulaict/skinova-ai-knowledge-base/Skinova_knowledge_base/WHO/who_common_skin_diseases.pdf
/kaggle/input/datasets/midhulaict/skinova-ai-knowledge-base/Skinova_knowledge_base/Dermnet/dermnet_seborrhoeic_keratosis.pdf
/kaggle/input/datasets/midhulaict/skinova-ai-knowledge-base/Skinova_knowledge_base/Dermnet/dermnet_bcc.pdf
/kaggle/input/datasets/midhulaict/skinova-ai-knowledge-base/Skinova_knowledge_base/Dermnet/dermnet_dermatofibroma.pdf
/kaggle/input/datasets/midhulaict/skinova-ai-knowledge-base/Skinova_knowledge_base/Dermnet/dermnet_actinic_keratosis.pdf
/kaggle/input/datasets/midhulaict/skinova-ai-knowledge-base/Skinova_knowledge_base/Dermnet/dermnet_melanoma.pdf
/kaggle/input/datasets/midhulaict/skinova-ai-knowledge-base/Skinova_knowledge_base/Dermnet/dermnet_vascular_lesions.pdf
/kaggle/input/datasets/midhulaict/skinova-ai-knowledge-base/Skinova_knowledge_base/Dermnet/dermnet_melanocytic_naevi.pdf


# Extract text from DermNet File using OCR

In [6]:
import glob
import os

pdf_files = glob.glob("/kaggle/input/**/*.pdf", recursive=True)

print("Total PDF files found:", len(pdf_files))
print("=" * 70)

for i, path in enumerate(pdf_files, 1):
    print(i, os.path.basename(path))

Total PDF files found: 8
1 who_common_skin_diseases.pdf
2 dermnet_seborrhoeic_keratosis.pdf
3 dermnet_bcc.pdf
4 dermnet_dermatofibroma.pdf
5 dermnet_actinic_keratosis.pdf
6 dermnet_melanoma.pdf
7 dermnet_vascular_lesions.pdf
8 dermnet_melanocytic_naevi.pdf


In [7]:
dermnet_files = [
    path for path in pdf_files
    if os.path.basename(path).startswith("dermnet_")
]

print("DermNet PDFs:", len(dermnet_files))
print("=" * 70)

for path in dermnet_files:
    print(os.path.basename(path))

DermNet PDFs: 7
dermnet_seborrhoeic_keratosis.pdf
dermnet_bcc.pdf
dermnet_dermatofibroma.pdf
dermnet_actinic_keratosis.pdf
dermnet_melanoma.pdf
dermnet_vascular_lesions.pdf
dermnet_melanocytic_naevi.pdf


In [8]:
import pymupdf
import pytesseract
from PIL import Image
import io
import os

# Find the melanoma PDF
melanoma_path = next(
    p for p in dermnet_files
    if os.path.basename(p) == "dermnet_melanoma.pdf"
)

# Open PDF
doc = pymupdf.open(melanoma_path)

print("File:", os.path.basename(melanoma_path))
print("Total pages:", len(doc))
print("=" * 70)

# OCR first 2 pages only
for page_num in range(min(2, len(doc))):
    
    page = doc[page_num]
    
    # Render PDF page as image at 2x resolution
    pix = page.get_pixmap(
        matrix=pymupdf.Matrix(2, 2),
        alpha=False
    )
    
    # Convert to PIL image
    image = Image.open(
        io.BytesIO(pix.tobytes("png"))
    )
    
    # OCR
    text = pytesseract.image_to_string(image)
    
    print(f"\n--- PAGE {page_num + 1} ---")
    print(text[:3000])

File: dermnet_melanoma.pdf
Total pages: 14

--- PAGE 1 ---
 

LESIONS (CANCEROUS)

Melanoma

October 2022

Author: Dr Nicole A. Seebacher, Department of Oncology, University of Oxford, United Kingdom, Ux. (2022)
Previous contributors: Dr Amanda Oakley, Dermatologist (1997)
Reviewing dermatologist: Dr lan Coulson

Edited by the DermNet content department

What is melanoma?

Melanoma, also referred to as malignant melanoma, is a potentially very serious skin cancer in which
there is an uncontrolled growth of melanocytes (pigment cells).

Normal melanocytes are found in the basal layer of the epidermis (outer layer of skin). Melanocytes
produce a protein called melanin, which protects skin cells by absorbing ultraviolet (UV) radiation.

Non-cancerous growth of melanocytes results in moles (benign melanocytic naevi) and freckles
(ephelides and lentigines). In contrast, the cancerous growth of melanocytes results in melanoma.
Melanoma is described as:

In situ, if a tumour is confined to th

In [9]:
import pymupdf
import pytesseract
from PIL import Image
import io
import os
import glob

# Folder to save OCR text
ocr_dir = "/kaggle/working/dermnet_ocr"
os.makedirs(ocr_dir, exist_ok=True)

print("Starting OCR...")
print("=" * 70)

for pdf_path in dermnet_files:

    filename = os.path.basename(pdf_path)
    output_name = os.path.splitext(filename)[0] + ".txt"
    output_path = os.path.join(ocr_dir, output_name)

    print(f"\nProcessing: {filename}")

    doc = pymupdf.open(pdf_path)
    all_text = []

    for page_num in range(len(doc)):

        page = doc[page_num]

        # Render page at 2x resolution
        pix = page.get_pixmap(
            matrix=pymupdf.Matrix(2, 2),
            alpha=False
        )

        # Convert to image
        image = Image.open(
            io.BytesIO(pix.tobytes("png"))
        )

        # OCR
        text = pytesseract.image_to_string(image)

        # Store page text
        all_text.append(
            f"\n--- PAGE {page_num + 1} ---\n{text}"
        )

        print(f"  Page {page_num + 1}/{len(doc)} done")

    # Save complete OCR text
    with open(output_path, "w", encoding="utf-8") as f:
        f.write("\n".join(all_text))

    print(f"Saved: {output_name}")

print("\n" + "=" * 70)
print("OCR COMPLETE")
print("Output folder:", ocr_dir)

Starting OCR...

Processing: dermnet_seborrhoeic_keratosis.pdf
  Page 1/6 done
  Page 2/6 done
  Page 3/6 done
  Page 4/6 done
  Page 5/6 done
  Page 6/6 done
Saved: dermnet_seborrhoeic_keratosis.txt

Processing: dermnet_bcc.pdf
  Page 1/10 done
  Page 2/10 done
  Page 3/10 done
  Page 4/10 done
  Page 5/10 done
  Page 6/10 done
  Page 7/10 done
  Page 8/10 done
  Page 9/10 done
  Page 10/10 done
Saved: dermnet_bcc.txt

Processing: dermnet_dermatofibroma.pdf
  Page 1/3 done
  Page 2/3 done
  Page 3/3 done
Saved: dermnet_dermatofibroma.txt

Processing: dermnet_actinic_keratosis.pdf
  Page 1/6 done
  Page 2/6 done
  Page 3/6 done
  Page 4/6 done
  Page 5/6 done
  Page 6/6 done
Saved: dermnet_actinic_keratosis.txt

Processing: dermnet_melanoma.pdf
  Page 1/14 done
  Page 2/14 done
  Page 3/14 done
  Page 4/14 done
  Page 5/14 done
  Page 6/14 done
  Page 7/14 done
  Page 8/14 done
  Page 9/14 done
  Page 10/14 done
  Page 11/14 done
  Page 12/14 done
  Page 13/14 done
  Page 14/14 done
Sa

# Check OCR Quality

In [10]:
ocr_files = sorted(
    glob.glob("/kaggle/working/dermnet_ocr/*.txt")
)

print("OCR text files:", len(ocr_files))
print("=" * 70)

for path in ocr_files:

    with open(path, "r", encoding="utf-8") as f:
        text = f.read()

    # Basic statistics
    characters = len(text)
    words = len(text.split())

    print(f"{os.path.basename(path)}")
    print(f"  Characters : {characters:,}")
    print(f"  Words      : {words:,}")
    print()

OCR text files: 7
dermnet_actinic_keratosis.txt
  Characters : 9,440
  Words      : 1,470

dermnet_bcc.txt
  Characters : 12,468
  Words      : 1,819

dermnet_dermatofibroma.txt
  Characters : 3,583
  Words      : 515

dermnet_melanocytic_naevi.txt
  Characters : 13,758
  Words      : 2,172

dermnet_melanoma.txt
  Characters : 25,410
  Words      : 3,821

dermnet_seborrhoeic_keratosis.txt
  Characters : 7,204
  Words      : 1,034

dermnet_vascular_lesions.txt
  Characters : 5,320
  Words      : 719



# Extract WHO file

In [15]:
!pip install pypdf -q


In [18]:
import glob
from pypdf import PdfReader
# Find WHO PDF
who_files = [
    path for path in glob.glob("/kaggle/input/**/*.pdf", recursive=True)
    if os.path.basename(path) == "who_common_skin_diseases.pdf"
]

if not who_files:
    raise FileNotFoundError("WHO PDF not found.")

who_path = who_files[0]

print("WHO file:", os.path.basename(who_path))

# Read PDF
reader = PdfReader(who_path)

print("Total pages:", len(reader.pages))
print("=" * 70)

# Extract text
who_text = []

for page_num, page in enumerate(reader.pages, 1):
    text = page.extract_text() or ""
    
    who_text.append(
        f"\n--- PAGE {page_num} ---\n{text}"
    )

    print(f"Page {page_num}/{len(reader.pages)} extracted")

# Combine
who_text = "\n".join(who_text)

# Save
who_output = "/kaggle/working/who_common_skin_diseases.txt"

with open(who_output, "w", encoding="utf-8") as f:
    f.write(who_text)

print("\n" + "=" * 70)
print("WHO EXTRACTION COMPLETE")
print("Characters:", len(who_text))
print("Words:", len(who_text.split()))
print("Saved to:", who_output)

WHO file: who_common_skin_diseases.pdf
Total pages: 45
Page 1/45 extracted
Page 2/45 extracted
Page 3/45 extracted
Page 4/45 extracted
Page 5/45 extracted
Page 6/45 extracted
Page 7/45 extracted
Page 8/45 extracted
Page 9/45 extracted
Page 10/45 extracted
Page 11/45 extracted
Page 12/45 extracted
Page 13/45 extracted
Page 14/45 extracted
Page 15/45 extracted
Page 16/45 extracted
Page 17/45 extracted
Page 18/45 extracted
Page 19/45 extracted
Page 20/45 extracted
Page 21/45 extracted
Page 22/45 extracted
Page 23/45 extracted
Page 24/45 extracted
Page 25/45 extracted
Page 26/45 extracted
Page 27/45 extracted
Page 28/45 extracted
Page 29/45 extracted
Page 30/45 extracted
Page 31/45 extracted
Page 32/45 extracted
Page 33/45 extracted
Page 34/45 extracted
Page 35/45 extracted
Page 36/45 extracted
Page 37/45 extracted
Page 38/45 extracted
Page 39/45 extracted
Page 40/45 extracted
Page 41/45 extracted
Page 42/45 extracted
Page 43/45 extracted
Page 44/45 extracted
Page 45/45 extracted

WHO EXTR

# Inspect the extracted text

In [19]:
print("=" * 80)
print("WHO TEXT SAMPLE")
print("=" * 80)

with open(
    "/kaggle/working/who_common_skin_diseases.txt",
    "r",
    encoding="utf-8"
) as f:
    who_text = f.read()

print(who_text[:5000])


# --------------------------------------------------
# 2. Inspect one DermNet text
# --------------------------------------------------

print("\n\n" + "=" * 80)
print("DERMNET MELANOMA TEXT SAMPLE")
print("=" * 80)

melanoma_txt = "/kaggle/working/dermnet_ocr/dermnet_melanoma.txt"

with open(
    melanoma_txt,
    "r",
    encoding="utf-8"
) as f:
    melanoma_text = f.read()

print(melanoma_text[:5000])

WHO TEXT SAMPLE

--- PAGE 1 ---
 
 
 
 
Common Skin Diseases for Management or Referral 
at Primary Health Care Level 
WHO  guidance with ICD -11 Mortality and Morbidity Statistics (MMS) codes, release 
2026 -01 
Version 1.0 
 
 
For external expert review draft 
 
 
  

--- PAGE 2 ---
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
Disclaimer 
This document has been prepared to support the recognition, management and referral of common skin diseases at the 
primary health care (PHC) level. It is intended to assist Ministries of Health, programme managers, educators and primary 
health care providers in strengthening integrated skin health services. 
The classification supports clinical decision-making, training, service delivery and health information systems. It should be 
adapted to national epidemiology, available resources and existing clinical guidelines. 
This document does not replace national clinical guidelines. Countries should adapt its recommend

# Clean and structure the extracted documents

In [20]:
import re

# ============================================================
# CLEAN EXTRACTED TEXT
# ============================================================

WHO_FILE = "/kaggle/working/who_common_skin_diseases.txt"
DERMNET_DIR = "/kaggle/working/dermnet_ocr"
CLEAN_DIR = "/kaggle/working/clean_text"

os.makedirs(CLEAN_DIR, exist_ok=True)


def clean_text(text):
    """
    Clean OCR/PDF extracted text while preserving
    the actual medical content.
    """

    # Normalize line endings
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # Remove form-feed characters
    text = text.replace("\f", "\n")

    # Remove repeated whitespace at line ends
    text = re.sub(r"[ \t]+$", "", text, flags=re.MULTILINE)

    # Fix words broken across lines by hyphenation
    # Example:
    # melan-
    # oma
    # -> melanoma
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)

    # Collapse excessive blank lines
    text = re.sub(r"\n{3,}", "\n\n", text)

    # Remove isolated page-number lines such as "1/14", "3/14"
    text = re.sub(r"^\s*\d+\s*/\s*\d+\s*$", "", text, flags=re.MULTILINE)

    # Remove lines containing only a page number
    text = re.sub(r"^\s*\d+\s*$", "", text, flags=re.MULTILINE)

    # Clean excessive spaces
    text = re.sub(r"[ \t]{2,}", " ", text)

    # Final whitespace cleanup
    text = text.strip()

    return text


# ------------------------------------------------------------
# Clean WHO
# ------------------------------------------------------------

with open(WHO_FILE, "r", encoding="utf-8") as f:
    who_text = f.read()

who_clean = clean_text(who_text)

who_output = os.path.join(
    CLEAN_DIR,
    "who_common_skin_diseases_clean.txt"
)

with open(who_output, "w", encoding="utf-8") as f:
    f.write(who_clean)


# ------------------------------------------------------------
# Clean DermNet files
# ------------------------------------------------------------

dermnet_files = sorted(
    f for f in os.listdir(DERMNET_DIR)
    if f.endswith(".txt")
)

for filename in dermnet_files:

    input_path = os.path.join(DERMNET_DIR, filename)

    with open(input_path, "r", encoding="utf-8") as f:
        text = f.read()

    cleaned = clean_text(text)

    output_filename = filename.replace(
        ".txt",
        "_clean.txt"
    )

    output_path = os.path.join(
        CLEAN_DIR,
        output_filename
    )

    with open(output_path, "w", encoding="utf-8") as f:
        f.write(cleaned)


# ------------------------------------------------------------
# Show results
# ------------------------------------------------------------

print("=" * 70)
print("CLEANING COMPLETE")
print("=" * 70)

print(f"Output directory: {CLEAN_DIR}")
print()

for filename in sorted(os.listdir(CLEAN_DIR)):

    path = os.path.join(CLEAN_DIR, filename)

    with open(path, "r", encoding="utf-8") as f:
        text = f.read()

    print(f"{filename}")
    print(f"  Characters : {len(text):,}")
    print(f"  Words      : {len(text.split()):,}")
    print()

CLEANING COMPLETE
Output directory: /kaggle/working/clean_text

dermnet_actinic_keratosis_clean.txt
  Characters : 9,367
  Words      : 1,464

dermnet_bcc_clean.txt
  Characters : 12,328
  Words      : 1,808

dermnet_dermatofibroma_clean.txt
  Characters : 3,554
  Words      : 513

dermnet_melanocytic_naevi_clean.txt
  Characters : 13,616
  Words      : 2,163

dermnet_melanoma_clean.txt
  Characters : 25,255
  Words      : 3,807

dermnet_seborrhoeic_keratosis_clean.txt
  Characters : 7,142
  Words      : 1,026

dermnet_vascular_lesions_clean.txt
  Characters : 5,242
  Words      : 714

who_common_skin_diseases_clean.txt
  Characters : 101,409
  Words      : 14,220



# Create source metadata

In [21]:
import json

# ============================================================
# CREATE SOURCE METADATA
# ============================================================

CLEAN_DIR = "/kaggle/working/clean_text"
METADATA_FILE = "/kaggle/working/source_metadata.json"

sources = [
    {
        "filename": "who_common_skin_diseases_clean.txt",
        "source": "WHO",
        "source_type": "clinical_guidance",
        "title": "Common Skin Diseases for Management or Referral at Primary Health Care Level",
        "disease": "Common skin diseases",
        "ham10000_class": None
    },
    {
        "filename": "dermnet_melanoma_clean.txt",
        "source": "DermNet",
        "source_type": "dermatology_reference",
        "title": "Melanoma",
        "disease": "Melanoma",
        "ham10000_class": "MEL"
    },
    {
        "filename": "dermnet_melanocytic_naevi_clean.txt",
        "source": "DermNet",
        "source_type": "dermatology_reference",
        "title": "Melanocytic naevi",
        "disease": "Melanocytic nevus",
        "ham10000_class": "NV"
    },
    {
        "filename": "dermnet_bcc_clean.txt",
        "source": "DermNet",
        "source_type": "dermatology_reference",
        "title": "Basal cell carcinoma",
        "disease": "Basal cell carcinoma",
        "ham10000_class": "BCC"
    },
    {
        "filename": "dermnet_actinic_keratosis_clean.txt",
        "source": "DermNet",
        "source_type": "dermatology_reference",
        "title": "Actinic keratosis",
        "disease": "Actinic keratosis",
        "ham10000_class": "AKIEC"
    },
    {
        "filename": "dermnet_seborrhoeic_keratosis_clean.txt",
        "source": "DermNet",
        "source_type": "dermatology_reference",
        "title": "Seborrhoeic keratosis",
        "disease": "Benign keratosis",
        "ham10000_class": "BKL"
    },
    {
        "filename": "dermnet_dermatofibroma_clean.txt",
        "source": "DermNet",
        "source_type": "dermatology_reference",
        "title": "Dermatofibroma",
        "disease": "Dermatofibroma",
        "ham10000_class": "DF"
    },
    {
        "filename": "dermnet_vascular_lesions_clean.txt",
        "source": "DermNet",
        "source_type": "dermatology_reference",
        "title": "Vascular lesions",
        "disease": "Vascular lesions",
        "ham10000_class": "VASC"
    }
]

# Check that every expected file exists
print("Checking source files...\n")

for item in sources:
    path = os.path.join(CLEAN_DIR, item["filename"])

    if os.path.exists(path):
        print("✓", item["filename"])
    else:
        print("✗ MISSING:", item["filename"])

# Save metadata
with open(METADATA_FILE, "w", encoding="utf-8") as f:
    json.dump(sources, f, indent=4, ensure_ascii=False)

print("\n" + "=" * 70)
print("SOURCE METADATA CREATED")
print("=" * 70)

print("Metadata file:")
print(METADATA_FILE)

print("\nTotal sources:", len(sources))

Checking source files...

✓ who_common_skin_diseases_clean.txt
✓ dermnet_melanoma_clean.txt
✓ dermnet_melanocytic_naevi_clean.txt
✓ dermnet_bcc_clean.txt
✓ dermnet_actinic_keratosis_clean.txt
✓ dermnet_seborrhoeic_keratosis_clean.txt
✓ dermnet_dermatofibroma_clean.txt
✓ dermnet_vascular_lesions_clean.txt

SOURCE METADATA CREATED
Metadata file:
/kaggle/working/source_metadata.json

Total sources: 8


# Create RAG chunks

In [22]:
from collections import Counter

# ============================================================
# CREATE RAG CHUNKS
# ============================================================

CLEAN_DIR = "/kaggle/working/clean_text"
METADATA_FILE = "/kaggle/working/source_metadata.json"
CHUNKS_FILE = "/kaggle/working/rag_chunks.json"

CHUNK_SIZE = 800
CHUNK_OVERLAP = 150


# ------------------------------------------------------------
# Load metadata
# ------------------------------------------------------------

with open(METADATA_FILE, "r", encoding="utf-8") as f:
    sources = json.load(f)


# ------------------------------------------------------------
# Normalize text
# ------------------------------------------------------------

def normalize_for_chunking(text):

    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # Remove excessive blank lines
    text = re.sub(r"\n{3,}", "\n\n", text)

    # Remove excessive spaces/tabs
    text = re.sub(r"[ \t]+", " ", text)

    return text.strip()


# ------------------------------------------------------------
# Create overlapping chunks
# ------------------------------------------------------------

def create_chunks(text, chunk_size=800, overlap=150):

    words = text.split()

    chunks = []

    start = 0
    chunk_number = 0

    while start < len(words):

        end = min(start + chunk_size, len(words))

        chunk_words = words[start:end]

        chunk_text = " ".join(chunk_words).strip()

        if chunk_text:

            chunks.append({
                "chunk_number": chunk_number,
                "text": chunk_text,
                "word_count": len(chunk_words)
            })

        # Stop after final chunk
        if end >= len(words):
            break

        # Move forward while retaining overlap
        start = end - overlap

        chunk_number += 1

    return chunks


# ------------------------------------------------------------
# Generate chunks
# ------------------------------------------------------------

all_chunks = []

for source in sources:

    filename = source["filename"]

    path = os.path.join(
        CLEAN_DIR,
        filename
    )

    if not os.path.exists(path):
        print(f"WARNING: Missing file: {filename}")
        continue

    with open(path, "r", encoding="utf-8") as f:
        text = f.read()

    text = normalize_for_chunking(text)

    source_chunks = create_chunks(
        text,
        chunk_size=CHUNK_SIZE,
        overlap=CHUNK_OVERLAP
    )

    for chunk in source_chunks:

        chunk_id = (
            f"{source['source'].lower()}_"
            f"{source['ham10000_class'] or 'GENERAL'}_"
            f"{chunk['chunk_number']:04d}"
        )

        all_chunks.append({

            # Unique identifier
            "chunk_id": chunk_id,

            # Original file
            "filename": filename,

            # Source information
            "source": source["source"],
            "source_type": source["source_type"],
            "title": source["title"],
            "disease": source["disease"],

            # HAM10000 mapping
            "ham10000_class": source["ham10000_class"],

            # Chunk information
            "chunk_number": chunk["chunk_number"],
            "word_count": chunk["word_count"],

            # Actual retrieved text
            "text": chunk["text"]
        })


# ------------------------------------------------------------
# Save chunks
# ------------------------------------------------------------

with open(CHUNKS_FILE, "w", encoding="utf-8") as f:

    json.dump(
        all_chunks,
        f,
        indent=2,
        ensure_ascii=False
    )


# ------------------------------------------------------------
# Statistics
# ------------------------------------------------------------

print("=" * 70)
print("RAG CHUNKING COMPLETE")
print("=" * 70)

print(f"Total chunks : {len(all_chunks):,}")
print(f"Chunk size   : {CHUNK_SIZE} words")
print(f"Overlap      : {CHUNK_OVERLAP} words")


# ------------------------------------------------------------
# Chunks by source
# ------------------------------------------------------------

print("\nChunks by source:")
print("-" * 70)

source_counts = Counter(
    chunk["title"]
    for chunk in all_chunks
)

for source in sources:

    title = source["title"]

    print(
        f"{source['source']:8s} | "
        f"{title:55s} | "
        f"{source_counts[title]:3d}"
    )


# ------------------------------------------------------------
# Word-count statistics
# ------------------------------------------------------------

word_counts = [
    chunk["word_count"]
    for chunk in all_chunks
]

if word_counts:

    print("\nChunk word-count statistics:")
    print("-" * 70)

    print(f"Minimum : {min(word_counts)}")
    print(f"Maximum : {max(word_counts)}")
    print(f"Average : {sum(word_counts) / len(word_counts):.1f}")


# ------------------------------------------------------------
# Verify chunk structure
# ------------------------------------------------------------

print("\nFirst chunk:")
print("-" * 70)

if all_chunks:

    first = all_chunks[0]

    print("chunk_id       :", first["chunk_id"])
    print("filename       :", first["filename"])
    print("source         :", first["source"])
    print("title          :", first["title"])
    print("disease        :", first["disease"])
    print("HAM10000 class :", first["ham10000_class"])
    print("word_count     :", first["word_count"])

    print("\nText preview:")
    print(first["text"][:1000])


print("\nSaved to:")
print(CHUNKS_FILE)

RAG CHUNKING COMPLETE
Total chunks : 42
Chunk size   : 800 words
Overlap      : 150 words

Chunks by source:
----------------------------------------------------------------------
WHO      | Common Skin Diseases for Management or Referral at Primary Health Care Level |  22
DermNet  | Melanoma                                                |   6
DermNet  | Melanocytic naevi                                       |   4
DermNet  | Basal cell carcinoma                                    |   3
DermNet  | Actinic keratosis                                       |   3
DermNet  | Seborrhoeic keratosis                                   |   2
DermNet  | Dermatofibroma                                          |   1
DermNet  | Vascular lesions                                        |   1

Chunk word-count statistics:
----------------------------------------------------------------------
Minimum : 164
Maximum : 800
Average : 733.7

First chunk:
--------------------------------------------------------

# Inspect chunks for retrieval quality

In [23]:
# ============================================================
# INSPECT RAG CHUNKS
# ============================================================

CHUNKS_FILE = "/kaggle/working/rag_chunks.json"

with open(CHUNKS_FILE, "r", encoding="utf-8") as f:
    chunks = json.load(f)


# ------------------------------------------------------------
# Helper function
# ------------------------------------------------------------

def show_chunk(chunk):

    print("=" * 80)
    print("CHUNK ID       :", chunk["chunk_id"])
    print("SOURCE         :", chunk["source"])
    print("TITLE          :", chunk["title"])
    print("DISEASE        :", chunk["disease"])
    print("HAM10000 CLASS :", chunk["ham10000_class"])
    print("CHUNK NUMBER   :", chunk["chunk_number"])
    print("WORD COUNT     :", chunk["word_count"])
    print("-" * 80)
    print(chunk["text"][:2500])
    print()


# ------------------------------------------------------------
# 1. First WHO chunk
# ------------------------------------------------------------

print("\n### WHO — FIRST CHUNK ###\n")

who_chunks = [
    c for c in chunks
    if c["source"] == "WHO"
]

show_chunk(who_chunks[0])


# ------------------------------------------------------------
# 2. WHO middle chunk
# ------------------------------------------------------------

print("\n### WHO — MIDDLE CHUNK ###\n")

show_chunk(who_chunks[len(who_chunks) // 2])


# ------------------------------------------------------------
# 3. WHO final chunk
# ------------------------------------------------------------

print("\n### WHO — FINAL CHUNK ###\n")

show_chunk(who_chunks[-1])


# ------------------------------------------------------------
# 4. Melanoma chunk
# ------------------------------------------------------------

print("\n### DERMNET — MELANOMA ###\n")

melanoma_chunks = [
    c for c in chunks
    if c["ham10000_class"] == "MEL"
]

show_chunk(melanoma_chunks[1])


# ------------------------------------------------------------
# 5. BCC chunk
# ------------------------------------------------------------

print("\n### DERMNET — BCC ###\n")

bcc_chunks = [
    c for c in chunks
    if c["ham10000_class"] == "BCC"
]

show_chunk(bcc_chunks[0])


# ------------------------------------------------------------
# 6. Actinic keratosis chunk
# ------------------------------------------------------------

print("\n### DERMNET — ACTINIC KERATOSIS ###\n")

akiec_chunks = [
    c for c in chunks
    if c["ham10000_class"] == "AKIEC"
]

show_chunk(akiec_chunks[0])


### WHO — FIRST CHUNK ###

CHUNK ID       : who_GENERAL_0000
SOURCE         : WHO
TITLE          : Common Skin Diseases for Management or Referral at Primary Health Care Level
DISEASE        : Common skin diseases
HAM10000 CLASS : None
CHUNK NUMBER   : 0
WORD COUNT     : 800
--------------------------------------------------------------------------------
--- PAGE 1 --- Common Skin Diseases for Management or Referral at Primary Health Care Level WHO guidance with ICD -11 Mortality and Morbidity Statistics (MMS) codes, release 2026 -01 Version 1.0 For external expert review draft --- PAGE 2 --- Disclaimer This document has been prepared to support the recognition, management and referral of common skin diseases at the primary health care (PHC) level. It is intended to assist Ministries of Health, programme managers, educators and primary health care providers in strengthening integrated skin health services. The classification supports clinical decision-making, training, service deliver

# Improve the chunking

In [24]:
INPUT_DIR = "/kaggle/working/clean_text"
OUTPUT_FILE = "/kaggle/working/rag_chunks_v2.json"

# Target size
CHUNK_SIZE = 700
OVERLAP = 120

with open("/kaggle/working/source_metadata.json", "r") as f:
    metadata = json.load(f)

# ---------------------------------------------------------
# Helper functions
# ---------------------------------------------------------

def clean_paragraph(text):
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def split_into_paragraphs(text):
    # Split on blank lines first
    paragraphs = re.split(r"\n\s*\n+", text)

    cleaned = []
    for p in paragraphs:
        p = clean_paragraph(p)
        if p:
            cleaned.append(p)

    return cleaned


def word_count(text):
    return len(text.split())


# ---------------------------------------------------------
# Load metadata by filename
# ---------------------------------------------------------

metadata_by_file = {
    item["filename"]: item
    for item in metadata
}


# ---------------------------------------------------------
# Create section-aware chunks
# ---------------------------------------------------------

all_chunks = []

for filename in sorted(os.listdir(INPUT_DIR)):

    if not filename.endswith(".txt"):
        continue

    filepath = os.path.join(INPUT_DIR, filename)

    with open(filepath, "r", encoding="utf-8") as f:
        text = f.read()

    paragraphs = split_into_paragraphs(text)

    source_info = metadata_by_file[filename]

    current = []
    current_words = 0
    chunk_number = 0

    for paragraph in paragraphs:

        p_words = word_count(paragraph)

        # Skip extremely tiny fragments
        if p_words < 5:
            continue

        # If adding paragraph exceeds target,
        # save current chunk first.
        if current and current_words + p_words > CHUNK_SIZE:

            chunk_text = "\n\n".join(current)

            chunk_id = (
                f"{source_info['ham10000_class'] or 'GENERAL'}"
                f"_{chunk_number:04d}"
            )

            all_chunks.append({
                "chunk_id": chunk_id,
                "filename": filename,
                "source": source_info["source"],
                "title": source_info["title"],
                "disease": source_info["disease"],
                "ham10000_class": source_info["ham10000_class"],
                "chunk_number": chunk_number,
                "word_count": word_count(chunk_text),
                "text": chunk_text
            })

            chunk_number += 1

            # -------------------------------------------------
            # Keep overlap using complete paragraphs
            # -------------------------------------------------

            overlap_paragraphs = []
            overlap_words = 0

            for old_p in reversed(current):

                old_words = word_count(old_p)

                if overlap_words + old_words > OVERLAP:
                    break

                overlap_paragraphs.insert(0, old_p)
                overlap_words += old_words

            current = overlap_paragraphs
            current_words = overlap_words

        current.append(paragraph)
        current_words += p_words

    # ---------------------------------------------------------
    # Save final chunk
    # ---------------------------------------------------------

    if current:

        chunk_text = "\n\n".join(current)

        chunk_id = (
            f"{source_info['ham10000_class'] or 'GENERAL'}"
            f"_{chunk_number:04d}"
        )

        all_chunks.append({
            "chunk_id": chunk_id,
            "filename": filename,
            "source": source_info["source"],
            "title": source_info["title"],
            "disease": source_info["disease"],
            "ham10000_class": source_info["ham10000_class"],
            "chunk_number": chunk_number,
            "word_count": word_count(chunk_text),
            "text": chunk_text
        })


# ---------------------------------------------------------
# Save
# ---------------------------------------------------------

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(all_chunks, f, ensure_ascii=False, indent=2)


# ---------------------------------------------------------
# Statistics
# ---------------------------------------------------------

print("SECTION-AWARE CHUNKING COMPLETE")
print("=" * 60)

print(f"Total chunks : {len(all_chunks)}")
print(f"Target size  : {CHUNK_SIZE} words")
print(f"Overlap      : {OVERLAP} words")

print("\nChunks by source:")

source_counts = {}

for chunk in all_chunks:
    source = chunk["source"]
    source_counts[source] = source_counts.get(source, 0) + 1

for source, count in source_counts.items():
    print(f"{source:20s}: {count}")

sizes = [c["word_count"] for c in all_chunks]

print("\nChunk word-count statistics:")
print(f"Minimum : {min(sizes)}")
print(f"Maximum : {max(sizes)}")
print(f"Average : {sum(sizes)/len(sizes):.1f}")

print(f"\nSaved to: {OUTPUT_FILE}")

SECTION-AWARE CHUNKING COMPLETE
Total chunks : 46
Target size  : 700 words
Overlap      : 120 words

Chunks by source:
DermNet             : 21
WHO                 : 25

Chunk word-count statistics:
Minimum : 212
Maximum : 758
Average : 605.6

Saved to: /kaggle/working/rag_chunks_v2.json


# Inspect the new chunks

In [25]:
CHUNK_FILE = "/kaggle/working/rag_chunks_v2.json"

with open(CHUNK_FILE, "r", encoding="utf-8") as f:
    chunks = json.load(f)

print("=" * 80)
print("RAG CHUNK QUALITY INSPECTION")
print("=" * 80)


def show_chunk(index):
    c = chunks[index]

    print("\n" + "=" * 80)
    print(f"INDEX        : {index}")
    print(f"CHUNK ID     : {c['chunk_id']}")
    print(f"SOURCE       : {c['source']}")
    print(f"DISEASE      : {c['disease']}")
    print(f"HAM10000     : {c['ham10000_class']}")
    print(f"WORD COUNT   : {c['word_count']}")
    print("-" * 80)
    print(c["text"][:3500])


# ---------------------------------------------------------
# 1. First WHO chunk
# ---------------------------------------------------------
print("\n### WHO — FIRST CHUNK")
show_chunk(0)


# ---------------------------------------------------------
# 2. Middle WHO chunk
# ---------------------------------------------------------
who_indices = [
    i for i, c in enumerate(chunks)
    if c["source"] == "WHO"
]

middle_who = who_indices[len(who_indices) // 2]

print("\n### WHO — MIDDLE CHUNK")
show_chunk(middle_who)


# ---------------------------------------------------------
# 3. Last WHO chunk
# ---------------------------------------------------------
print("\n### WHO — FINAL CHUNK")
show_chunk(who_indices[-1])


# ---------------------------------------------------------
# 4. Melanoma chunk
# ---------------------------------------------------------
mel_indices = [
    i for i, c in enumerate(chunks)
    if c["ham10000_class"] == "MEL"
]

print("\n### DERMNET — MELANOMA")
show_chunk(mel_indices[0])


# ---------------------------------------------------------
# 5. BCC chunk
# ---------------------------------------------------------
bcc_indices = [
    i for i, c in enumerate(chunks)
    if c["ham10000_class"] == "BCC"
]

print("\n### DERMNET — BCC")
show_chunk(bcc_indices[0])


# ---------------------------------------------------------
# 6. Actinic keratosis
# ---------------------------------------------------------
ak_indices = [
    i for i, c in enumerate(chunks)
    if c["ham10000_class"] == "AKIEC"
]

print("\n### DERMNET — ACTINIC KERATOSIS")
show_chunk(ak_indices[0])

RAG CHUNK QUALITY INSPECTION

### WHO — FIRST CHUNK

INDEX        : 0
CHUNK ID     : AKIEC_0000
SOURCE       : DermNet
DISEASE      : Actinic keratosis
HAM10000     : AKIEC
WORD COUNT   : 685
--------------------------------------------------------------------------------
Authors: Dr lan Coulson, Dermatologist, United Kingdom; Editor-in-Chief of DermNet (2024). Minor update November 2025. Previous contributors: Dr Amanda Oakley, Dermatologist, Hamilton, New Zealand, (1897); further updated December 2015.

Edited by the DermNet content department.

What is an actinic keratosis?

Actinic keratosis is a precancerous scaly spot found on sun-damaged skin, also known as solar keratosis. It may be considered an early form of cutaneous squamous cell carcinoma (a keratinocyte cancer).

& f oe Te Whatu Ora

Apink base with a hyperkeratotic top on the nasal bridge-a An actinic keratosis on the nose common site for actinic keratoses

A rough scally lesion on the back of the hand - a common site fo

# Install the embedding/vector-search libraries

In [26]:
!pip install -q sentence-transformers faiss-cpu

In [27]:
import sentence_transformers
import faiss

print("Sentence Transformers:", sentence_transformers.__version__)
print("FAISS:", faiss.__version__)

print("\nEmbedding/vector libraries ready.")

Sentence Transformers: 5.4.1
FAISS: 1.15.1

Embedding/vector libraries ready.


# Load the embedding model

In [28]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

print("Loading embedding model...")
embedding_model = SentenceTransformer(MODEL_NAME)

print("\nEmbedding model loaded successfully.")
print("Model:", MODEL_NAME)
print("Embedding dimension:", embedding_model.get_sentence_embedding_dimension())

Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


Embedding model loaded successfully.
Model: sentence-transformers/all-MiniLM-L6-v2
Embedding dimension: 384


/tmp/ipykernel_58/1708439071.py:10: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimension:", embedding_model.get_sentence_embedding_dimension())


# Generate embeddings for all 46 chunks

In [29]:
import numpy as np

# Load our section-aware RAG chunks
CHUNKS_PATH = "/kaggle/working/rag_chunks_v2.json"

with open(CHUNKS_PATH, "r", encoding="utf-8") as f:
    chunks = json.load(f)

print("Loaded chunks:", len(chunks))

# Extract chunk text
texts = [chunk["text"] for chunk in chunks]

print("Generating embeddings...")

embeddings = embedding_model.encode(
    texts,
    batch_size=16,
    show_progress_bar=True,
    normalize_embeddings=True
)

embeddings = np.asarray(embeddings, dtype="float32")

print("\nEMBEDDINGS GENERATED")
print("Shape:", embeddings.shape)
print("Data type:", embeddings.dtype)
print("Min value:", embeddings.min())
print("Max value:", embeddings.max())

Loaded chunks: 46
Generating embeddings...


Batches:   0%|          | 0/3 [00:00<?, ?it/s]


EMBEDDINGS GENERATED
Shape: (46, 384)
Data type: float32
Min value: -0.20128012
Max value: 0.21577203


# Build the FAISS vector index

In [30]:
import faiss

# Make sure embeddings are float32
embeddings = np.asarray(embeddings, dtype="float32")

# Create FAISS index using inner product.
# Because embeddings were normalized, inner product = cosine similarity.
embedding_dim = embeddings.shape[1]

index = faiss.IndexFlatIP(embedding_dim)
index.add(embeddings)

print("FAISS INDEX CREATED")
print("Embedding dimension:", embedding_dim)
print("Vectors in index:", index.ntotal)
print("Index type:", type(index).__name__)

# Save index
INDEX_PATH = "/kaggle/working/skinova_faiss.index"
faiss.write_index(index, INDEX_PATH)

print("\nSaved to:", INDEX_PATH)
print("File exists:", os.path.exists(INDEX_PATH))
print("File size (MB):", round(os.path.getsize(INDEX_PATH) / (1024**2), 3))

FAISS INDEX CREATED
Embedding dimension: 384
Vectors in index: 46
Index type: IndexFlatIP

Saved to: /kaggle/working/skinova_faiss.index
File exists: True
File size (MB): 0.067


# Test the RAG retrieval

In [31]:


def search_rag(query, top_k=5):
    """
    Search the FAISS index and return the most relevant knowledge chunks.
    """
    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = index.search(query_embedding, top_k)

    results = []

    for score, idx in zip(scores[0], indices[0]):
        chunk = chunks[int(idx)]

        results.append({
            "score": float(score),
            "source": chunk["source"],
            "filename": chunk["filename"],
            "ham10000_class": chunk.get("ham10000_class"),
            "text": chunk["text"]
        })

    return results


# Test query
query = "What are the clinical features and risk factors of melanoma?"

results = search_rag(query, top_k=5)

print("=" * 80)
print("QUERY:", query)
print("=" * 80)

for i, result in enumerate(results, 1):
    print(f"\nRESULT {i}")
    print("-" * 80)
    print("Similarity score :", round(result["score"], 4))
    print("Source           :", result["source"])
    print("HAM10000 class   :", result["ham10000_class"])
    print("Filename         :", result["filename"])
    print("\nText:")
    print(result["text"][:1200])

QUERY: What are the clinical features and risk factors of melanoma?

RESULT 1
--------------------------------------------------------------------------------
Similarity score : 0.6852
Source           : DermNet
HAM10000 class   : MEL
Filename         : dermnet_melanoma_clean.txt

Text:
--- PAGE 13 --- Most tests are not worthwhile for patients with stage | or 2 melanoma unless there are signs or symptoms of disease recurrence or metastasis. No tests are necessary for healthy patients who have remained well for five years or longer after the removal of their melanoma.

How do you prevent melanoma?

Preventative measures involve addressing risk factors such as exposure to UV radiation, eg, wearing protective clothing, using sunscreen (SPF 50), and avoiding tanning beds. For more information, see skin

What is the outcome of melanoma?

Melanoma in situ is cured by excision because it has no potential to spread around the body.

The risk of spread and ultimate death from invasive melanoma

# Test retrieval across all 7 classes

In [32]:
test_queries = {
    "MEL": "What are the clinical features and risk factors of melanoma?",
    "NV": "What are the clinical features and characteristics of melanocytic naevi?",
    "BCC": "What are the clinical features and risk factors of basal cell carcinoma?",
    "AKIEC": "What are the clinical features and risk factors of actinic keratosis?",
    "BKL": "What are the clinical features and characteristics of seborrhoeic keratosis?",
    "DF": "What are the clinical features and characteristics of dermatofibroma?",
    "VASC": "What are the clinical features and characteristics of vascular lesions?"
}

for expected_class, query in test_queries.items():

    results = search_rag(query, top_k=3)

    print("\n" + "=" * 90)
    print(f"EXPECTED CLASS: {expected_class}")
    print(f"QUERY: {query}")
    print("=" * 90)

    for i, result in enumerate(results, 1):
        print(
            f"{i}. "
            f"Score={result['score']:.4f} | "
            f"Class={result['ham10000_class']} | "
            f"Source={result['source']}"
        )


EXPECTED CLASS: MEL
QUERY: What are the clinical features and risk factors of melanoma?
1. Score=0.6852 | Class=MEL | Source=DermNet
2. Score=0.6594 | Class=MEL | Source=DermNet
3. Score=0.6387 | Class=MEL | Source=DermNet

EXPECTED CLASS: NV
QUERY: What are the clinical features and characteristics of melanocytic naevi?
1. Score=0.6804 | Class=NV | Source=DermNet
2. Score=0.6559 | Class=MEL | Source=DermNet
3. Score=0.6478 | Class=NV | Source=DermNet

EXPECTED CLASS: BCC
QUERY: What are the clinical features and risk factors of basal cell carcinoma?
1. Score=0.6925 | Class=BCC | Source=DermNet
2. Score=0.6634 | Class=BCC | Source=DermNet
3. Score=0.5333 | Class=None | Source=WHO

EXPECTED CLASS: AKIEC
QUERY: What are the clinical features and risk factors of actinic keratosis?
1. Score=0.6898 | Class=AKIEC | Source=DermNet
2. Score=0.6856 | Class=AKIEC | Source=DermNet
3. Score=0.5293 | Class=BKL | Source=DermNet

EXPECTED CLASS: BKL
QUERY: What are the clinical features and characte

# Save the complete RAG knowledge base

In [33]:

RAG_DIR = "/kaggle/working/skinova_rag"
os.makedirs(RAG_DIR, exist_ok=True)

# --------------------------------------------------
# 1. Save chunks
# --------------------------------------------------

chunks_path = os.path.join(RAG_DIR, "chunks.json")

with open(chunks_path, "w", encoding="utf-8") as f:
    json.dump(chunks, f, ensure_ascii=False, indent=2)


# --------------------------------------------------
# 2. Save embeddings
# --------------------------------------------------

embeddings_path = os.path.join(RAG_DIR, "embeddings.npy")

np.save(
    embeddings_path,
    np.asarray(embeddings, dtype="float32")
)


# --------------------------------------------------
# 3. Save FAISS index
# --------------------------------------------------

index_path = os.path.join(RAG_DIR, "faiss.index")

faiss.write_index(index, index_path)


# --------------------------------------------------
# 4. Save source metadata
# --------------------------------------------------

metadata_source = "/kaggle/working/source_metadata.json"
metadata_path = os.path.join(RAG_DIR, "source_metadata.json")

with open(metadata_source, "r", encoding="utf-8") as f:
    source_metadata = json.load(f)

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(source_metadata, f, ensure_ascii=False, indent=2)


# --------------------------------------------------
# 5. Save model information
# --------------------------------------------------

model_info = {
    "embedding_model": MODEL_NAME,
    "embedding_dimension": int(embeddings.shape[1]),
    "normalization": "L2 normalized",
    "similarity": "cosine similarity via inner product",
    "num_chunks": len(chunks),
    "num_vectors": int(index.ntotal)
}

model_info_path = os.path.join(RAG_DIR, "model_info.json")

with open(model_info_path, "w", encoding="utf-8") as f:
    json.dump(model_info, f, indent=2)


# --------------------------------------------------
# Verify
# --------------------------------------------------

print("=" * 70)
print("SKINOVA RAG KNOWLEDGE BASE SAVED")
print("=" * 70)

for filename in sorted(os.listdir(RAG_DIR)):
    filepath = os.path.join(RAG_DIR, filename)
    size_mb = os.path.getsize(filepath) / (1024 ** 2)
    print(f"{filename:<25} {size_mb:.3f} MB")

print("\nChunks :", len(chunks))
print("Vectors:", index.ntotal)
print("Dimension:", embeddings.shape[1])
print("\nRAG directory:", RAG_DIR)

SKINOVA RAG KNOWLEDGE BASE SAVED
chunks.json               0.201 MB
embeddings.npy            0.068 MB
faiss.index               0.067 MB
model_info.json           0.000 MB
source_metadata.json      0.002 MB

Chunks : 46
Vectors: 46
Dimension: 384

RAG directory: /kaggle/working/skinova_rag


# Create the retrieve() function

In [34]:
def retrieve(query, top_k=5, min_score=0.0):
    """
    Retrieve the most relevant knowledge chunks for a query.

    Parameters
    ----------
    query : str
        User/clinical question.
    top_k : int
        Number of chunks to retrieve.
    min_score : float
        Optional minimum cosine similarity score.

    Returns
    -------
    list
        Retrieved chunks with scores and source information.
    """

    # Encode query
    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    ).astype("float32")

    # Search FAISS
    scores, indices = index.search(query_embedding, top_k)

    results = []

    for score, idx in zip(scores[0], indices[0]):

        # Ignore invalid FAISS indices
        if idx < 0:
            continue

        score = float(score)

        # Optional score filtering
        if score < min_score:
            continue

        chunk = chunks[int(idx)]

        results.append({
            "rank": len(results) + 1,
            "score": round(score, 4),
            "source": chunk["source"],
            "filename": chunk["filename"],
            "ham10000_class": chunk.get("ham10000_class"),
            "text": chunk["text"]
        })

    return results


print("retrieve() function created successfully.")

retrieve() function created successfully.


# Create the RAG context formatter

In [35]:
def format_rag_context(results):
    """
    Convert retrieved RAG results into a structured context
    that can be passed to an LLM.
    """

    if not results:
        return "No relevant knowledge was retrieved."

    context_parts = []

    for result in results:
        part = f"""
SOURCE {result['rank']}
Source: {result['source']}
Document: {result['filename']}
HAM10000 class: {result['ham10000_class']}
Similarity: {result['score']}

Knowledge:
{result['text']}
"""
        context_parts.append(part.strip())

    return "\n\n" + "\n\n".join(context_parts)


# Test the formatter
query = "What are the clinical features of basal cell carcinoma?"

results = retrieve(query, top_k=3)

context = format_rag_context(results)

print("=" * 80)
print("RAG CONTEXT")
print("=" * 80)
print(context[:6000])

RAG CONTEXT


SOURCE 1
Source: DermNet
Document: dermnet_bcc_clean.txt
HAM10000 class: BCC
Similarity: 0.6773

Knowledge:
What are the complications of basal cell carcinoma?

Recurrence of BCC after initial treatment is not uncommon. Characteristics of recurrent BCC often include:

Incomplete excision or narrow margins at primary excision Morphoeic, micronodular, and infiltrative subtypes Location on head and neck.

After PDT After superficial surgery

--- PAGE 6 --- Te Whatu Ora

Advanced BCCs are large, often neglected tumours.

They may be several centimetres in diameter They may be deeply infiltrating into tissues below the skin They are difficult or impossible to treat surgically

Primary tumour is often large, neglected or recurrent, located on head and neck, with aggressive subtype

May have had multiple prior treatments

May arise in-site exposed to ionising radiation Can be fatal

How is basal cell carcinoma diagnosed?

BCC is diagnosed clinically by the presence of a slowly e

# Class-aware retrieval

In [36]:
def retrieve_class_aware(query, predicted_class=None, top_k=5, candidate_k=15):
    """
    Retrieve knowledge using both semantic similarity and,
    when available, the predicted HAM10000 class.

    predicted_class:
        One of MEL, NV, BCC, AKIEC, BKL, DF, VASC
        or None.
    """

    # Encode query
    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    ).astype("float32")

    # Retrieve a larger candidate pool first
    scores, indices = index.search(
        query_embedding,
        min(candidate_k, index.ntotal)
    )

    candidates = []

    for score, idx in zip(scores[0], indices[0]):

        if idx < 0:
            continue

        chunk = chunks[int(idx)]

        candidates.append({
            "original_score": float(score),
            "source": chunk["source"],
            "filename": chunk["filename"],
            "ham10000_class": chunk.get("ham10000_class"),
            "text": chunk["text"]
        })

    # If no class is supplied, use normal semantic ranking
    if predicted_class is None:
        candidates.sort(
            key=lambda x: x["original_score"],
            reverse=True
        )

    else:
        # Give a small ranking bonus to chunks belonging
        # to the predicted HAM10000 class.
        #
        # We do NOT remove other sources/classes because
        # WHO/general medical knowledge may still be useful.
        for candidate in candidates:

            if candidate["ham10000_class"] == predicted_class:
                candidate["ranking_score"] = (
                    candidate["original_score"] + 0.05
                )
            else:
                candidate["ranking_score"] = candidate["original_score"]

        candidates.sort(
            key=lambda x: x["ranking_score"],
            reverse=True
        )

    # Return top results
    results = []

    for rank, candidate in enumerate(
        candidates[:top_k], 1
    ):
        results.append({
            "rank": rank,
            "score": round(candidate["original_score"], 4),
            "source": candidate["source"],
            "filename": candidate["filename"],
            "ham10000_class": candidate["ham10000_class"],
            "text": candidate["text"]
        })

    return results


print("Class-aware retrieval function created successfully.")

Class-aware retrieval function created successfully.


In [37]:
### Test

query = "What are the clinical features and risk factors of basal cell carcinoma?"

predicted_class = "BCC"

results = retrieve_class_aware(
    query=query,
    predicted_class=predicted_class,
    top_k=5
)

print("=" * 80)
print("QUERY:", query)
print("PREDICTED CLASS:", predicted_class)
print("=" * 80)

for result in results:
    print(
        f"\nRank {result['rank']} | "
        f"Score: {result['score']} | "
        f"Class: {result['ham10000_class']} | "
        f"Source: {result['source']}"
    )

QUERY: What are the clinical features and risk factors of basal cell carcinoma?
PREDICTED CLASS: BCC

Rank 1 | Score: 0.6925 | Class: BCC | Source: DermNet

Rank 2 | Score: 0.6634 | Class: BCC | Source: DermNet

Rank 3 | Score: 0.5333 | Class: None | Source: WHO

Rank 4 | Score: 0.4878 | Class: MEL | Source: DermNet

Rank 5 | Score: 0.4708 | Class: MEL | Source: DermNet


# Build the final evidence package

In [38]:
def build_evidence_package(
    query,
    predicted_class,
    prediction_confidence=None,
    top_k=5
):
    """
    Build a structured evidence package for the Skinova LLM.

    Parameters
    ----------
    query : str
        User's question or explanation request.

    predicted_class : str
        EfficientNetB1 predicted HAM10000 class.

    prediction_confidence : float, optional
        Model confidence, e.g. 0.87.

    top_k : int
        Number of RAG chunks to retrieve.
    """

    # Retrieve class-aware medical knowledge
    retrieved = retrieve_class_aware(
        query=query,
        predicted_class=predicted_class,
        top_k=top_k
    )

    # Format retrieved evidence
    context = format_rag_context(retrieved)

    # Build structured package
    package = {
        "query": query,
        "prediction": {
            "ham10000_class": predicted_class,
            "confidence": prediction_confidence
        },
        "retrieval": {
            "num_results": len(retrieved),
            "results": retrieved
        },
        "llm_context": context
    }

    return package


# --------------------------------------------------
# Test the evidence package
# --------------------------------------------------

evidence = build_evidence_package(
    query="What are the clinical features and risk factors of basal cell carcinoma?",
    predicted_class="BCC",
    prediction_confidence=0.91,
    top_k=5
)

print("=" * 80)
print("SKINOVA EVIDENCE PACKAGE")
print("=" * 80)

print("\nPrediction:")
print(evidence["prediction"])

print("\nRetrieved sources:")

for result in evidence["retrieval"]["results"]:
    print(
        f"  {result['rank']}. "
        f"{result['source']} | "
        f"{result['ham10000_class']} | "
        f"score={result['score']}"
    )

print("\nLLM context preview:")
print(evidence["llm_context"][:5000])

SKINOVA EVIDENCE PACKAGE

Prediction:
{'ham10000_class': 'BCC', 'confidence': 0.91}

Retrieved sources:
  1. DermNet | BCC | score=0.6925
  2. DermNet | BCC | score=0.6634
  3. WHO | None | score=0.5333
  4. DermNet | MEL | score=0.4878
  5. DermNet | MEL | score=0.4708

LLM context preview:


SOURCE 1
Source: DermNet
Document: dermnet_bcc_clean.txt
HAM10000 class: BCC
Similarity: 0.6925

Knowledge:
--- PAGE 1 --- == DermNet°®

Authors: Honorary Associate Profes Anthony Martin Fuentes, General Practitioner and Cosmetic

Update peer reviewed by: Dr Andje bourne Hospital, Australia, (2028)

Reviewing dermatologist: Dr lan Ce Edited by the DermNet content department.

What is basal cell carcinoma?

Basal cell carcinoma (BCC) is a common, locally invasive, keratinocyte cancer (also known as nonmelanoma cancer). It is the most common form of skin cancer. BCC is also known as rodent ulcer and basalioma. Patients with BCC often develop multiple primary tumours over time.

BCC is also known as

# Create the Skinova LLM prompt

In [39]:
def build_skinova_prompt(evidence):
    """
    Build the final grounded prompt for the Skinova LLM.
    """

    prediction = evidence["prediction"]
    retrieved = evidence["retrieval"]["results"]

    predicted_class = prediction["ham10000_class"]
    confidence = prediction["confidence"]

    # Convert confidence to readable form
    if confidence is not None:
        confidence_text = f"{confidence:.2%}"
    else:
        confidence_text = "Not provided"

    # Build source evidence
    evidence_blocks = []

    for result in retrieved:
        evidence_blocks.append(
            f"""
SOURCE {result['rank']}
Source: {result['source']}
Document: {result['filename']}
HAM10000 class tag: {result['ham10000_class']}
Similarity score: {result['score']}

{result['text']}
""".strip()
        )

    evidence_text = "\n\n".join(evidence_blocks)

    prompt = f"""
You are Skinova, an AI-assisted dermatology support system.

Your task is to provide a clear, cautious, knowledge-grounded explanation
based on an image classification result and retrieved medical references.

IMAGE CLASSIFICATION RESULT
Predicted HAM10000 class: {predicted_class}
Classifier confidence: {confidence_text}

IMPORTANT:
- The image classifier prediction is an AI prediction, not a confirmed diagnosis.
- Do not state that the patient definitely has the predicted condition.
- Do not invent clinical findings that are not present in the supplied evidence.
- Use the retrieved medical sources to explain the predicted class.
- Prioritize retrieved evidence whose HAM10000 class tag matches the predicted class.
- WHO/general evidence may be used when relevant.
- Do not use evidence tagged with a different disease class as if it were
  evidence about the predicted class.
- If the retrieved sources do not support a claim, say that the available
  sources do not provide enough information.
- Do not fabricate citations, statistics, treatments, or recommendations.
- Clearly distinguish the classifier prediction from information contained
  in the medical references.
- For medical decisions, emphasize that appropriate clinical assessment
  and professional evaluation are required.

RETRIEVED MEDICAL EVIDENCE
==========================

{evidence_text}

RESPONSE REQUIREMENTS
=====================

Answer the user's question using the evidence above.

Structure the response as appropriate, using sections such as:

1. AI prediction
2. What the retrieved medical sources say
3. Relevant clinical features
4. Relevant risk factors or considerations
5. Important limitations

Keep the explanation understandable to a non-specialist while retaining
medically important terminology.

Do not present the AI prediction as a definitive diagnosis.
""".strip()

    return prompt


# --------------------------------------------------
# Test prompt generation
# --------------------------------------------------

skinova_prompt = build_skinova_prompt(evidence)

print("=" * 80)
print("SKINOVA LLM PROMPT")
print("=" * 80)
print(skinova_prompt[:10000])

SKINOVA LLM PROMPT
You are Skinova, an AI-assisted dermatology support system.

Your task is to provide a clear, cautious, knowledge-grounded explanation
based on an image classification result and retrieved medical references.

IMAGE CLASSIFICATION RESULT
Predicted HAM10000 class: BCC
Classifier confidence: 91.00%

IMPORTANT:
- The image classifier prediction is an AI prediction, not a confirmed diagnosis.
- Do not state that the patient definitely has the predicted condition.
- Do not invent clinical findings that are not present in the supplied evidence.
- Use the retrieved medical sources to explain the predicted class.
- Prioritize retrieved evidence whose HAM10000 class tag matches the predicted class.
- WHO/general evidence may be used when relevant.
- Do not use evidence tagged with a different disease class as if it were
  evidence about the predicted class.
- If the retrieved sources do not support a claim, say that the available
  sources do not provide enough information.
-

# Choose and connect the LLM

In [40]:
import torch

In [41]:
print("=" * 70)
print("SKINOVA LLM ENVIRONMENT")
print("=" * 70)

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2),
        "GB"
    )
else:
    print("GPU: Not available")

print("\nPython:", os.sys.version.split()[0])

SKINOVA LLM ENVIRONMENT
PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB

Python: 3.12.13


In [42]:
print("=" * 70)
print("SKINOVA LLM ENVIRONMENT")
print("=" * 70)

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))
    print(
        f"GPU {i} memory:",
        round(torch.cuda.get_device_properties(i).total_memory / (1024**3), 2),
        "GB"
    )

print("\nPython:", os.sys.version.split()[0])

SKINOVA LLM ENVIRONMENT
PyTorch version: 2.10.0+cu128
CUDA available: True
GPU count: 2
GPU 0: Tesla T4
GPU 0 memory: 14.56 GB
GPU 1: Tesla T4
GPU 1 memory: 14.56 GB

Python: 3.12.13


# Install the LLM packages

In [43]:
!pip install -q -U transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 86.9 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 42.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 85.8 MB/s eta 0:00:00:00:01


# Verify the LLM packages

In [44]:
import transformers
import accelerate
import bitsandbytes
import torch

print("=" * 70)
print("SKINOVA LLM PACKAGES")
print("=" * 70)

print("Transformers:", transformers.__version__)
print("Accelerate:", accelerate.__version__)
print("BitsAndBytes:", bitsandbytes.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

SKINOVA LLM PACKAGES
Transformers: 5.0.0
Accelerate: 1.13.0
BitsAndBytes: 0.50.2
CUDA available: True
GPU count: 2


# Download and load Qwen2.5-7B-Instruct

In [54]:
!pip install -q --force-reinstall \
    "transformers==4.57.1" \
    "huggingface_hub==0.36.0" \
    "tokenizers>=0.22,<0.24"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.9/45.9 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 88.9 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.1/566.1 kB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 89.8 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.6/206.6 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 93.0 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 86.0 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.0/130.0 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 807.9/807.9 kB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━

In [55]:
import huggingface_hub
print("=" * 70)
print("SKINOVA LLM ENVIRONMENT — FINAL CHECK")
print("=" * 70)

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Hugging Face Hub:", huggingface_hub.__version__)
print("Accelerate:", accelerate.__version__)
print("BitsAndBytes:", bitsandbytes.__version__)

print("\nCUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

SKINOVA LLM ENVIRONMENT — FINAL CHECK
PyTorch: 2.10.0+cu128
Transformers: 5.0.0
Hugging Face Hub: 1.11.0
Accelerate: 1.13.0
BitsAndBytes: 0.50.2

CUDA available: True
GPU count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4


In [56]:
!pip install -q --target=/kaggle/working/skinova_llm_env \
    "transformers==4.57.1" \
    "huggingface_hub==0.36.0" \
    "tokenizers>=0.22,<0.24"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
datasets 5.0.0 requires fsspec[http]<=2026.4.0,>=2023.1.0, but you have fsspec 2026.7.0 which is incompatible.
ydata-profiling 4.18.4 requires numpy<2.4,>=1.22, but you have numpy 2.5.3 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2

In [57]:
import os

llm_env = "/kaggle/working/skinova_llm_env"

print("=" * 70)
print("SKINOVA ISOLATED LLM ENVIRONMENT")
print("=" * 70)

print("Environment exists:",
      os.path.exists(llm_env))

print("Transformers package:",
      os.path.exists(os.path.join(llm_env, "transformers")))

print("Hugging Face Hub package:",
      os.path.exists(os.path.join(llm_env, "huggingface_hub")))

print("Tokenizers package:",
      os.path.exists(os.path.join(llm_env, "tokenizers")))

SKINOVA ISOLATED LLM ENVIRONMENT
Environment exists: True
Transformers package: True
Hugging Face Hub package: True
Tokenizers package: True


# Activate the isolated LLM packages